# 02 — Data Cleaning

**Input:** `data/raw_combined.csv` (34,435 rows from notebook 01)

**Cleaning steps:**
1. Remove exact duplicate rows (5)
2. Remove garbage reviews: repeated-char spam, no alphabetic content (3)
3. Normalize whitespace (collapse multiple newlines, strip)
4. Strip URLs from review text
5. Replace "Loading..." translations with NaN (mark for re-translation)
6. Parse dates to datetime
7. Keep short reviews and English-in-French as-is (valid data, just edge cases)
8. Keep PII as-is (not harmful for NLP, note for later if needed)

**Output:** `data/cleaned.csv`

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'raw_combined.csv')
print(f'Loaded raw_combined.csv: {len(df):,} rows x {df.shape[1]} cols')
initial_count = len(df)

Loaded raw_combined.csv: 34,435 rows x 10 cols


---
## Step 1 — Remove exact duplicate rows

In [2]:
dedup_cols = [c for c in df.columns if c != 'source_file']
n_dupes = df.duplicated(subset=dedup_cols, keep='first').sum()
df = df.drop_duplicates(subset=dedup_cols, keep='first').reset_index(drop=True)
print(f'Removed {n_dupes} exact duplicate rows')
print(f'Rows: {len(df):,}')

Removed 5 exact duplicate rows
Rows: 34,430


---
## Step 2 — Remove garbage reviews

Only remove truly non-textual content: repeated-character spam (e.g. "XXXX...", ".......") and reviews with no alphabetic characters. Short reviews (<20 chars) are kept — they carry valid signal (e.g. "a fuir" = 1-star sentiment).

In [3]:
avis_str = df['avis'].astype(str)

# Repeated character reviews (<=3 unique chars, >5 chars long)
is_repeated_char = avis_str.apply(lambda x: len(set(x.strip())) <= 3 and len(x.strip()) > 5)

# No alphabetic content at all
is_no_alpha = ~avis_str.str.contains('[a-zA-Z\u00C0-\u00FF]', regex=True)

is_garbage = is_repeated_char | is_no_alpha
print(f'Garbage reviews found: {is_garbage.sum()}')
if is_garbage.sum() > 0:
    print('Removed:')
    for idx in df.loc[is_garbage].index:
        print(f'  [{idx}] "{str(df.loc[idx, "avis"])[:80]}"')

df = df[~is_garbage].reset_index(drop=True)
print(f'\nRows: {len(df):,}')

Garbage reviews found: 2
Removed:
  [10319] "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
  [33644] "......."

Rows: 34,428


---
## Step 3 — Normalize whitespace

In [4]:
def normalize_whitespace(text):
    if pd.isna(text):
        return text
    text = str(text)
    # Collapse 3+ newlines into 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Collapse runs of spaces/tabs (not newlines) into single space
    text = re.sub(r'[^\S\n]+', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

df['avis'] = df['avis'].apply(normalize_whitespace)
df['avis_en'] = df['avis_en'].apply(normalize_whitespace)

# Show before/after stats
newline_counts = df['avis'].astype(str).str.count(r'\n')
print(f'After normalization:')
print(f'  Max newlines in a review: {newline_counts.max()}')
print(f'  Reviews with >10 newlines: {(newline_counts > 10).sum()}')

After normalization:
  Max newlines in a review: 72
  Reviews with >10 newlines: 395


---
## Step 4 — Strip URLs

In [5]:
url_pattern = r'https?://\S+|www\.\S+'

has_url_before = df['avis'].astype(str).str.contains(url_pattern, regex=True).sum()
df['avis'] = df['avis'].astype(str).str.replace(url_pattern, '', regex=True).str.strip()
df['avis_en'] = df['avis_en'].astype(str).str.replace(url_pattern, '', regex=True).str.strip()
has_url_after = df['avis'].astype(str).str.contains(url_pattern, regex=True).sum()

print(f'URLs stripped: {has_url_before} reviews had URLs, {has_url_after} remain after cleaning')

URLs stripped: 31 reviews had URLs, 0 remain after cleaning


---
## Step 5 — Handle "Loading..." translations

Replace "Loading..." with NaN so downstream code can easily filter or re-translate these rows.

In [6]:
is_loading = df['avis_en'].astype(str).str.strip().str.lower().isin(['loading...', 'loading\u2026', 'loading'])
print(f'"Loading..." translations: {is_loading.sum():,} rows')
df.loc[is_loading, 'avis_en'] = np.nan
print(f'Replaced with NaN. Total avis_en NaN now: {df["avis_en"].isna().sum():,}')

"Loading..." translations: 1,104 rows
Replaced with NaN. Total avis_en NaN now: 1,106


---
## Step 6 — Parse dates

In [7]:
df['date_publication'] = pd.to_datetime(df['date_publication'], format='%d/%m/%Y', errors='coerce')
df['date_exp'] = pd.to_datetime(df['date_exp'], format='%d/%m/%Y', errors='coerce')

print(f'date_publication parsed: {df["date_publication"].notna().sum():,} / {len(df):,}')
print(f'date_exp parsed: {df["date_exp"].notna().sum():,} / {len(df):,}')
print(f'Date range: {df["date_publication"].min()} to {df["date_publication"].max()}')

date_publication parsed: 34,428 / 34,428
date_exp parsed: 34,428 / 34,428
Date range: 2016-11-17 00:00:00 to 2021-11-16 00:00:00


---
## Cleaning Summary

In [8]:
print(f'=== Cleaning Summary ===')
print(f'  Start:  {initial_count:,} rows')
print(f'  Final:  {len(df):,} rows')
print(f'  Removed: {initial_count - len(df):,} rows ({(initial_count - len(df))/initial_count*100:.2f}%)')
print(f'    - Exact duplicates: {n_dupes}')
print(f'    - Garbage reviews: {is_garbage.sum()}')
print()
print(f'  Text cleaning applied to all rows:')
print(f'    - Whitespace normalized')
print(f'    - URLs stripped from {has_url_before} reviews')
print(f'    - {is_loading.sum():,} "Loading..." translations replaced with NaN')
print(f'    - Dates parsed to datetime')
print()
print(f'  Not removed (kept as-is):')
print(f'    - Short reviews (<20 chars): still present, valid signal')
print(f'    - English-in-French reviews: ~19 rows, kept')
print(f'    - PII: not stripped (not harmful for NLP models)')
print()
print(f'  Columns: {list(df.columns)}')
print(f'  Dtypes:')
print(df.dtypes.to_string())

=== Cleaning Summary ===
  Start:  34,435 rows
  Final:  34,428 rows
  Removed: 7 rows (0.02%)
    - Exact duplicates: 5
    - Garbage reviews: 2

  Text cleaning applied to all rows:
    - Whitespace normalized
    - URLs stripped from 31 reviews
    - 1,104 "Loading..." translations replaced with NaN
    - Dates parsed to datetime

  Not removed (kept as-is):
    - Short reviews (<20 chars): still present, valid signal
    - English-in-French reviews: ~19 rows, kept
    - PII: not stripped (not harmful for NLP models)

  Columns: ['note', 'auteur', 'avis', 'assureur', 'produit', 'type', 'date_publication', 'date_exp', 'avis_en', 'source_file']
  Dtypes:
note                       float64
auteur                         str
avis                           str
assureur                       str
produit                        str
type                           str
date_publication    datetime64[us]
date_exp            datetime64[us]
avis_en                        str
source_file          

In [9]:
# Null counts after cleaning
print('=== Null counts ===')
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(null_df[null_df['null_count'] > 0].to_string())
print()
print(f'Train/test split: {df["note"].notna().sum():,} / {df["note"].isna().sum():,}')

=== Null counts ===


         null_count  null_pct
note          10329     30.00
auteur            1      0.00
avis_en        1106      3.21

Train/test split: 24,099 / 10,329


### Post-Cleaning Observations

**Short reviews (<20 chars, 107 rows):** These carry clear sentiment signal and were intentionally kept. Examples: "A fuir" (1-star), "Prix inegale" (4-star), "Quelle lenteur!" (2-star), "Pas de commentaire." (5-star). Removing them would lose valid training data.

**NaN avis_en (1,106 rows):** 1,104 are former "Loading..." translations (failed web scrape), 2 are original NaNs (file boundary glitch + trivial "Aucun" review). The French text (`avis`) is fully intact for all of them — only the English translation is missing. Downstream tasks using `avis` are unaffected; tasks using `avis_en` should filter these out.

**Very long reviews (>5000 chars, 12 rows):** These are real, detailed customer complaints — not garbled data. They may need truncation for transformer models with token limits, but the content is valid.

**NaN auteur (1 row):** The deleted-post placeholder ("Intervention supprimee a la demande de l'internaute"). Still has a 3-star rating and an insurer. Kept because it doesn't interfere with NLP tasks, but it's not a real review.

**General quality:** Random sampling shows clean text, proper whitespace, no residual URLs, consistent FR/EN alignment where translations exist. Translation quality is acceptable but imperfect (e.g. "resilient" instead of "cancel/terminate" for "resilier").

### Why Spelling Correction Was Initially Considered Unnecessary

**Why transformers can handle typos natively:**
1. **Subword tokenization** (WordPiece / BPE): Misspelled words are split into recognizable subword tokens, so models like CamemBERT still capture meaning.
2. **TF-IDF impact is minimal**: Misspelled words become rare tokens with very low weight.
3. **Risk of corrupting domain vocabulary**: Insurance jargon and insurer names could be "corrected" into wrong words — this is mitigated below with a domain whitelist.
4. **The original `avis_cor` column** was 98.7% NaN — even the dataset creators did not invest in spelling correction.

---
## Save

In [10]:
df.to_csv(DATA_DIR / 'cleaned.csv', index=False)
print(f'Saved data/cleaned.csv: {len(df):,} rows x {df.shape[1]} cols')

# Verify
check = pd.read_csv(DATA_DIR / 'cleaned.csv')
print(f'Reload check: {len(check):,} rows, columns={list(check.columns)}')

Saved data/cleaned.csv: 34,428 rows x 10 cols


Reload check: 34,428 rows, columns=['note', 'auteur', 'avis', 'assureur', 'produit', 'type', 'date_publication', 'date_exp', 'avis_en', 'source_file']


---
## Section A — Automatic Translation of Missing `avis_en` Rows

1,106 rows have `avis_en = NaN` (1,104 were "Loading..." from failed web scrapes, 2 were original NaN).
We use **MarianMT** (`Helsinki-NLP/opus-mt-fr-en`) — a fully local, offline FR→EN translation model — to fill these gaps.

### A.1 — Load MarianMT Model

`Helsinki-NLP/opus-mt-fr-en` — a compact FR→EN model (~300 MB, runs on CPU).

In [11]:
%pip install -q --break-system-packages sacremoses

from transformers import MarianMTModel, MarianTokenizer

model_name = 'Helsinki-NLP/opus-mt-fr-en'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()

print(f'Model:      {model_name}')
print(f'Max length: {tokenizer.model_max_length}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device:     {next(model.parameters()).device}')

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Model:      Helsinki-NLP/opus-mt-fr-en
Max length: 512
Parameters: 75,133,952
Device:     cpu


### A.2 — Translation Function with Chunking

~9 reviews exceed the 512-token limit. We split those by sentence boundaries, translate each chunk, then reassemble.

In [12]:
import re, torch

def split_into_chunks(text, tokenizer, max_tokens=450):
    """Split text into chunks that fit the model's token limit."""
    sentences = re.split(r'(?<=[.!?\n])\s+', text.strip())
    chunks, current = [], []
    current_len = 0

    for sent in sentences:
        sent_len = len(tokenizer.encode(sent, add_special_tokens=False))
        # If a single sentence exceeds the limit, split by words
        if sent_len > max_tokens:
            words = sent.split()
            sub_chunk = []
            sub_len = 0
            for w in words:
                w_len = len(tokenizer.encode(w, add_special_tokens=False))
                if sub_len + w_len > max_tokens and sub_chunk:
                    chunks.append(' '.join(sub_chunk))
                    sub_chunk, sub_len = [], 0
                sub_chunk.append(w)
                sub_len += w_len
            if sub_chunk:
                if current:
                    current.append(' '.join(sub_chunk))
                    current_len += sub_len
                else:
                    chunks.append(' '.join(sub_chunk))
            continue

        if current_len + sent_len > max_tokens and current:
            chunks.append(' '.join(current))
            current, current_len = [], 0
        current.append(sent)
        current_len += sent_len

    if current:
        chunks.append(' '.join(current))
    return chunks if chunks else [text]


def translate_texts(texts, tokenizer, model, batch_size=32):
    """Translate a list of French texts to English, handling long texts via chunking."""
    import time as _time
    t0 = _time.time()
    texts = list(texts)
    # Build (index, chunk) pairs
    all_chunks = []
    chunk_counts = []
    for i, text in enumerate(texts):
        chunks = split_into_chunks(str(text), tokenizer)
        chunk_counts.append(len(chunks))
        for chunk in chunks:
            all_chunks.append(chunk)

    multi_chunk = sum(1 for c in chunk_counts if c > 1)
    print(f'Texts: {len(texts)}, Total chunks: {len(all_chunks)}, Multi-chunk texts: {multi_chunk}')

    # Sort chunks by length for efficient padding, keep original indices
    indexed_chunks = list(enumerate(all_chunks))
    indexed_chunks.sort(key=lambda x: len(x[1]))
    sort_order = [x[0] for x in indexed_chunks]
    sorted_chunks = [x[1] for x in indexed_chunks]

    # Batch translate all chunks (greedy decoding for speed)
    translated_sorted = [''] * len(sorted_chunks)
    total_batches = (len(sorted_chunks) + batch_size - 1) // batch_size
    for start in range(0, len(sorted_chunks), batch_size):
        batch = sorted_chunks[start:start + batch_size]
        encoded = tokenizer(batch, return_tensors='pt', padding=True, truncation=True,
                            max_length=tokenizer.model_max_length)
        with torch.no_grad():
            output = model.generate(**encoded, num_beams=1, do_sample=False)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        for j, d in enumerate(decoded):
            translated_sorted[start + j] = d
        batch_num = start // batch_size + 1
        if batch_num % 5 == 0 or batch_num == total_batches:
            elapsed = _time.time() - t0
            print(f'  Batch {batch_num}/{total_batches} ({elapsed:.0f}s)')

    # Unsort back to original order
    translated_chunks = [''] * len(all_chunks)
    for sorted_idx, orig_idx in enumerate(sort_order):
        translated_chunks[orig_idx] = translated_sorted[sorted_idx]

    # Reassemble multi-chunk translations
    translations = []
    idx = 0
    for count in chunk_counts:
        parts = translated_chunks[idx:idx + count]
        translations.append(' '.join(parts))
        idx += count

    print(f'  Total translation time: {_time.time() - t0:.0f}s')
    return translations

print('Translation functions defined.')

Translation functions defined.


### A.3 — Translate and Verify

Apply translation to the 1,106 NaN `avis_en` rows.

In [13]:
nan_mask = df['avis_en'].isna()
print(f'Rows to translate: {nan_mask.sum():,}')

texts_to_translate = df.loc[nan_mask, 'avis'].tolist()
translations = translate_texts(texts_to_translate, tokenizer, model, batch_size=16)

# Fill translations
df.loc[nan_mask, 'avis_en'] = translations

# Add provenance column
df['avis_en_source'] = 'website'
df.loc[nan_mask, 'avis_en_source'] = 'marianmt'

print(f'\nRemaining avis_en NaN: {df["avis_en"].isna().sum()}')
print(f'\nProvenance:')
print(df['avis_en_source'].value_counts().to_string())

# Before/after examples
print('\n--- Translation Examples ---')
sample_indices = df.loc[nan_mask].index[:6]
for i in sample_indices:
    fr = str(df.loc[i, 'avis'])[:120]
    en = str(df.loc[i, 'avis_en'])[:120]
    print(f'\n[{i}] FR: {fr}{"..." if len(str(df.loc[i,"avis"])) > 120 else ""}')
    print(f'     EN: {en}{"..." if len(str(df.loc[i,"avis_en"])) > 120 else ""}')

# Quality spot-check: length ratios
website_mask = df['avis_en_source'] == 'website'
marian_mask = df['avis_en_source'] == 'marianmt'
ratio_web = (df.loc[website_mask, 'avis_en'].str.len() / df.loc[website_mask, 'avis'].str.len()).median()
ratio_mt = (df.loc[marian_mask, 'avis_en'].str.len() / df.loc[marian_mask, 'avis'].str.len()).median()
print(f'\nEN/FR length ratio — website: {ratio_web:.2f}, marianmt: {ratio_mt:.2f}')

Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 512). Running this sequence through the model will result in indexing errors


Rows to translate: 1,106


Texts: 1106, Total chunks: 1119, Multi-chunk texts: 11


  Batch 5/70 (18s)


  Batch 10/70 (37s)


...

  Batch 50/70 (200s)


  Batch 55/70 (223s)


  Batch 60/70 (258s)


  Batch 65/70 (361s)


  Batch 70/70 (519s)
  Total translation time: 519s

Remaining avis_en NaN: 0

Provenance:
avis_en_source
website     33322
marianmt     1106

--- Translation Examples ---

[9999] FR: La rapidité du traitement de dossier, de l'envoi des documents nécessaires rapide et de l'amabilité des interlocuteurs a...
     EN: The speed of processing of the file, the dispatch of the necessary documents quickly and the friendliness of the interlo...

[13000] FR: J'ai fait un devis sur le site, devis établis à 236e, après appel d'un conseillé pour vérifier, le prix passe à 471e, sa...
     EN: I made a quote on the site, priced 236th, after calling a recommendation to check, the price goes up to 471st, for no re...

[13001] FR: Il est impossible d'avoir un interlocuteur sérieux pour avoir la position des comptes. Les informations demandées par in...
     EN: It is impossible to have a serious contact person to get the position of accounts. The information requested by the inte...

[13004] FR: Je sui

---
## Section B — Automated Spelling Correction

**Library:** `pyspellchecker` with `distance=1` (~15 sec vs 6.5 hours at distance=2)

**Strategy:** Three-layer whitelist to minimize false positives, then build a corpus-wide correction map and apply via fast regex substitution.

**Whitelist layers:**
1. **Domain terms** — insurer names, product names, insurance jargon (programmatic + curated)
2. **Hunspell French dictionary** — ~320k word forms via `spylls` (pure-Python hunspell), catches "mail", "pro", "etc", "sécu", etc.
3. **Corpus frequency threshold** — any word appearing ≥50 times in the corpus is assumed correct

**Primary impact:** Accent restoration (~90% of corrections: "tres"→"très", "vehicule"→"véhicule")

In [14]:
%pip install -q --break-system-packages pyspellchecker spylls

from spellchecker import SpellChecker

sp = SpellChecker(language='fr', distance=1)
print(f'SpellChecker loaded: language=fr, distance=1')
print(f'Dictionary size: {sp.word_frequency.unique_words:,} words')

Note: you may need to restart the kernel to use updated packages.


SpellChecker loaded: language=fr, distance=1
Dictionary size: 139,905 words


In [15]:
# === Whitelist Layer 1: Domain terms (insurer names, products, insurance jargon) ===
whitelist = set()

for col in ['assureur', 'produit']:
    for val in df[col].dropna().unique():
        for token in re.split(r'[\s\-\.\(\)/]+', str(val).lower()):
            if len(token) >= 2:
                whitelist.add(token)

curated = [
    # Contraction fragments
    'aujourd', 'jusqu', 'jai', 'lorsqu', 'presqu', 'quelqu', 'quoiqu', 'dun', 'dune',
    # Modern words
    'email', 'emails', 'covid', 'comparateur', 'comparateurs', 'internet', 'online', 'smartphone',
    # Insurance terms
    'hamon', 'resilier', 'resiliation', 'sinistre', 'sinistres', 'franchise', 'malus', 'bonus',
    'multirisque', 'tiers', 'courtier', 'courtiers', 'souscription', 'assureur', 'assureurs',
    'indemnisation', 'indemniser', 'surprime', 'avenant', 'avenants', 'cpam', 'rib',
    # Abbreviations and common informal
    'tel', 'sms', 'ok', 'bof', 'ras', 'rdv', 'lrar', 'ar',
    # Common proper nouns in reviews
    'trustpilot', 'google', 'facebook',
]
whitelist.update(curated)
print(f'Layer 1 — Domain whitelist: {len(whitelist)} words')

# === Whitelist Layer 2: Hunspell French dictionary (via spylls) ===
import urllib.request
from spylls.hunspell import Dictionary
import os

hunspell_dir = '/tmp/hunspell-fr'
os.makedirs(hunspell_dir, exist_ok=True)
base_url = 'https://raw.githubusercontent.com/LibreOffice/dictionaries/master/fr_FR/'
for fname in ['fr.dic', 'fr.aff']:
    dest = os.path.join(hunspell_dir, fname)
    if not os.path.exists(dest):
        urllib.request.urlretrieve(base_url + fname, dest)
hunspell_dict = Dictionary.from_files(os.path.join(hunspell_dir, 'fr'))
print(f'Layer 2 — Hunspell French dictionary loaded')

# === Whitelist Layer 3: Corpus frequency threshold ===
from collections import Counter

word_freq = Counter()
for text in df['avis'].dropna():
    word_freq.update(re.findall(r'[a-zA-Z\u00C0-\u00FF]+', str(text).lower()))

FREQ_THRESHOLD = 50
freq_whitelist = {w for w, c in word_freq.items() if c >= FREQ_THRESHOLD}
print(f'Layer 3 — Corpus frequency whitelist (>= {FREQ_THRESHOLD} occurrences): {len(freq_whitelist)} words')

# Load domain + frequency whitelists into spellchecker
sp.word_frequency.load_words(list(whitelist | freq_whitelist))
print(f'\nTotal whitelist loaded into SpellChecker: {len(whitelist | freq_whitelist):,} words')

Layer 1 — Domain whitelist: 134 words


Layer 2 — Hunspell French dictionary loaded


Layer 3 — Corpus frequency whitelist (>= 50 occurrences): 2557 words

Total whitelist loaded into SpellChecker: 2,600 words


In [16]:
import time

# Extract all unique words from corpus
all_words = set()
for text in df['avis'].dropna():
    all_words.update(re.findall(r'[a-zA-Z\u00C0-\u00FF]+', str(text).lower()))

print(f'Unique words in corpus: {len(all_words):,}')

# Find unknown words (after domain + frequency whitelists)
unknown = sp.unknown(all_words)
print(f'Unknown words (after domain + frequency whitelist): {len(unknown):,}')

# Layer 2: Filter out words known to Hunspell
start = time.time()
hunspell_known = {w for w in unknown if hunspell_dict.lookup(w)}
unknown_final = unknown - hunspell_known
hunspell_elapsed = time.time() - start
print(f'Hunspell recognized: {len(hunspell_known):,} — Unknown remaining: {len(unknown_final):,} ({hunspell_elapsed:.1f}s)')

# Build correction map (only truly unknown words with len >= 3)
start = time.time()
correction_map = {}
to_correct = [w for w in unknown_final if len(w) >= 3]
print(f'Words to correct (len >= 3): {len(to_correct):,}')

for w in to_correct:
    corrected = sp.correction(w)
    if corrected and corrected != w:
        correction_map[w] = corrected

elapsed = time.time() - start
print(f'\nCorrection map built in {elapsed:.1f}s')
print(f'Corrections found: {len(correction_map):,}')

Unique words in corpus: 35,045
Unknown words (after domain + frequency whitelist): 13,306


Hunspell recognized: 1,734 — Unknown remaining: 11,572 (25.4s)
Words to correct (len >= 3): 11,344



Correction map built in 11.4s
Corrections found: 7,564


In [17]:
# Top 30 corrections by corpus frequency (sanity check)
from collections import Counter

# Count word frequencies in corpus
word_freq = Counter()
for text in df['avis'].dropna():
    word_freq.update(re.findall(r'[a-zA-Z\u00C0-\u00FF]+', str(text).lower()))

# Sort corrections by frequency of the misspelled word
corrections_by_freq = sorted(
    correction_map.items(),
    key=lambda x: word_freq.get(x[0], 0),
    reverse=True
)

print('Top 30 corrections by corpus frequency:')
print(f'{"Original":<25} {"Corrected":<25} {"Frequency":>10}')
print('-' * 62)
for orig, corr in corrections_by_freq[:30]:
    print(f'{orig:<25} {corr:<25} {word_freq.get(orig, 0):>10,}')

Top 30 corrections by corpus frequency:
Original                  Corrected                  Frequency
--------------------------------------------------------------
clio                      clin                              48
reception                 réception                         48
declaration               déclaration                       48
malgre                    malgré                            47
delai                     délai                             47
uvre                      ouvre                             47
ald                       and                               46
celà                      cela                              46
tjs                       tes                               46
medecin                   médecin                           45
envoye                    envoyé                            43
cost                      coût                              43
repondu                   répondu                           43
end            

In [18]:
def apply_corrections(text, corr_map):
    """Apply spelling corrections while preserving case."""
    def replace_word(match):
        word = match.group(0)
        lower = word.lower()
        if lower not in corr_map:
            return word
        replacement = corr_map[lower]
        # Preserve case: ALL CAPS, Title Case, lowercase
        if word.isupper() and len(word) > 1:
            return replacement.upper()
        elif word[0].isupper():
            return replacement[0].upper() + replacement[1:]
        return replacement
    return re.sub(r'[a-zA-Z\u00C0-\u00FF]+', replace_word, str(text))

start = time.time()
df['avis_corrected'] = df['avis'].apply(lambda x: apply_corrections(x, correction_map))
elapsed = time.time() - start
print(f'Corrections applied to {len(df):,} rows in {elapsed:.1f}s')

Corrections applied to 34,428 rows in 2.1s


In [19]:
# Before/after examples
changed_mask = df['avis'] != df['avis_corrected']
changed_indices = df[changed_mask].index.tolist()
print(f'Reviews changed: {len(changed_indices):,} / {len(df):,} ({len(changed_indices)/len(df)*100:.1f}%)')

print('\n--- Before/After Examples ---')

# Find diverse examples
examples_shown = 0

# 1. Multiple accent restorations
for idx in changed_indices:
    orig = str(df.loc[idx, 'avis'])
    corr = str(df.loc[idx, 'avis_corrected'])
    diffs = sum(1 for a, b in zip(orig.split(), corr.split()) if a != b)
    if 3 <= diffs <= 8 and len(orig) < 400:
        print(f'\n[Example 1: Multiple corrections]')
        print(f'  BEFORE: {orig[:300]}')
        print(f'  AFTER:  {corr[:300]}')
        examples_shown += 1
        break

# 2. Actual spelling fix
for idx in changed_indices:
    orig = str(df.loc[idx, 'avis']).lower()
    if any(w in orig for w in ['rembourssement', 'rembourcement', 'remboursment', 'assurence', 'assurrence']):
        print(f'\n[Example 2: Spelling fix]')
        print(f'  BEFORE: {str(df.loc[idx, "avis"])[:300]}')
        print(f'  AFTER:  {str(df.loc[idx, "avis_corrected"])[:300]}')
        examples_shown += 1
        break

# 3. Clean review (no change)
unchanged = df[~changed_mask].index.tolist()
if unchanged:
    idx = unchanged[len(unchanged)//2]
    print(f'\n[Example 3: Clean review — no change]')
    print(f'  BEFORE: {str(df.loc[idx, "avis"])[:200]}')
    print(f'  AFTER:  {str(df.loc[idx, "avis_corrected"])[:200]}')

# 4. Review with insurer name preserved
for idx in changed_indices:
    text = str(df.loc[idx, 'avis'])
    assureur = str(df.loc[idx, 'assureur']).lower()
    if assureur in text.lower() and len(text) < 400:
        print(f'\n[Example 4: Insurer name preserved — "{df.loc[idx, "assureur"]}"]')
        print(f'  BEFORE: {text[:300]}')
        print(f'  AFTER:  {str(df.loc[idx, "avis_corrected"])[:300]}')
        examples_shown += 1
        break

# 5. ALL-CAPS review
for idx in changed_indices:
    text = str(df.loc[idx, 'avis'])
    if text == text.upper() and len(text) > 30:
        print(f'\n[Example 5: ALL-CAPS case preservation]')
        print(f'  BEFORE: {text[:300]}')
        print(f'  AFTER:  {str(df.loc[idx, "avis_corrected"])[:300]}')
        examples_shown += 1
        break

Reviews changed: 9,755 / 34,428 (28.3%)

--- Before/After Examples ---

[Example 1: Multiple corrections]
  BEFORE: ok bon deroulement du contrat et fluidite du site correcte et intuitive
tous les elements étaietn clairs et precis
je recommande cette assurance à mes proches
  AFTER:  ok bon déroulement du contrat et fluidité du site correcte et intuitive
tous les elements étaient clairs et précis
je recommande cette assurance à mes proches

[Example 2: Spelling fix]
  BEFORE: Tre bien je conseille cette assurence tre rapide son assurance en peux de temp tre appréciable de travailler avec des personne comme vous très cordialement
  AFTER:  Te bien je conseille cette assurance te rapide son assurance en peux de temps te appréciable de travailler avec des personne comme vous très cordialement

[Example 3: Clean review — no change]
  BEFORE: Après un bris de glace, en plus de la franchise attendez vous à une augmentation de tarif de 5,64 %.
En effet, malgré leur publicité télévisée de ne p

In [20]:
# Quantitative evaluation
changed_mask = df['avis'] != df['avis_corrected']
n_changed = changed_mask.sum()
print(f'=== Spelling Correction Summary ===')
print(f'  Reviews changed:     {n_changed:,} / {len(df):,} ({n_changed/len(df)*100:.1f}%)')
print(f'  Correction map size: {len(correction_map):,} entries')

# Count total word-level corrections
total_corrections = 0
for idx in df[changed_mask].index:
    orig_words = str(df.loc[idx, 'avis']).split()
    corr_words = str(df.loc[idx, 'avis_corrected']).split()
    if len(orig_words) == len(corr_words):
        total_corrections += sum(1 for a, b in zip(orig_words, corr_words) if a != b)
print(f'  Total word-level corrections: ~{total_corrections:,}')

# Random spot-checks
print('\n--- 5 Random Spot-Checks ---')
import random
random.seed(42)
spot = random.sample(changed_indices, min(5, len(changed_indices)))
for idx in spot:
    orig = str(df.loc[idx, 'avis'])
    corr = str(df.loc[idx, 'avis_corrected'])
    # Find changed words
    o_words = orig.split()
    c_words = corr.split()
    changes = []
    if len(o_words) == len(c_words):
        changes = [(a, b) for a, b in zip(o_words, c_words) if a != b]
    print(f'\n  [{idx}] Changes: {changes[:5]}{"..." if len(changes) > 5 else ""}')

=== Spelling Correction Summary ===
  Reviews changed:     9,755 / 34,428 (28.3%)
  Correction map size: 7,564 entries


  Total word-level corrections: ~19,150

--- 5 Random Spot-Checks ---

  [6423] Changes: [('CRM', 'CRU'), ('fevrier', 'février')]

  [1466] Changes: [('acc', 'arc')]

  [15950] Changes: [('Nice', 'Nièce'), ('Medecin', 'Médecin'), ('Nice.', 'Nièce.')]

  [14183] Changes: [('désœuvré,', 'désœouvré,')]

  [12953] Changes: [('end', 'en')]


### Limitations of Spelling Correction

- **distance=1 can't fix multi-accent words**: e.g. "deces" (should be "décès") — these need distance=2, which takes ~6.5 hours
- **Grammar errors untouched**: agreement errors, wrong conjugation, etc. are beyond scope
- **Hunspell + frequency threshold greatly reduce false positives**: "mail", "pro", "etc", "sécu", "perso" are now correctly preserved — but rare legitimate words unknown to both dictionaries may still be miscorrected
- **Practical impact on model performance likely small**: transformers handle typos natively via subword tokenization (BPE / WordPiece). The main value is cleaner exploratory analysis and TF-IDF features

In [21]:
# Re-save cleaned.csv with new columns
df.to_csv(DATA_DIR / 'cleaned.csv', index=False)
print(f'Saved data/cleaned.csv: {len(df):,} rows x {df.shape[1]} cols')
print(f'Columns: {list(df.columns)}')

# Verify on reload
check = pd.read_csv(DATA_DIR / 'cleaned.csv')
print(f'\nReload verification:')
print(f'  Rows: {len(check):,}')
print(f'  Columns ({len(check.columns)}): {list(check.columns)}')
print(f'  avis_en NaN: {check["avis_en"].isna().sum()}')
print(f'  avis_en_source values: {check["avis_en_source"].value_counts().to_dict()}')
print(f'  avis_corrected NaN: {check["avis_corrected"].isna().sum()}')

Saved data/cleaned.csv: 34,428 rows x 12 cols
Columns: ['note', 'auteur', 'avis', 'assureur', 'produit', 'type', 'date_publication', 'date_exp', 'avis_en', 'source_file', 'avis_en_source', 'avis_corrected']



Reload verification:
  Rows: 34,428
  Columns (12): ['note', 'auteur', 'avis', 'assureur', 'produit', 'type', 'date_publication', 'date_exp', 'avis_en', 'source_file', 'avis_en_source', 'avis_corrected']
  avis_en NaN: 1
  avis_en_source values: {'website': 33322, 'marianmt': 1106}
  avis_corrected NaN: 0
